# Banco Experimental de Pruebas: Clasificador de Residuos
## Aprendizaje Automático II - Práctica 1
Cuaderno de experimentación y evaluación empírica sistemática:
1. Pipeline de datos con aumento estocástico
2. Red convolucional propia (WasteCNN) y parada anticipada
3. Búsqueda automática de hiperparámetros con Optuna (TPE)
4. Transfer Learning con MobileNetV2 (Red ligera para edge computing)
5. Transfer Learning con ResNet-18 (Red residual para máxima precisión)
6. Análisis cuantitativo comparativo y exportación de métricas


## 0. Preparación del Entorno y Dataset
Carga automatizada del dataset en Google Colab o entorno local y creación del directorio de resultados.


In [ ]:
import os
import shutil

os.makedirs('results', exist_ok=True)

if not os.path.exists('Basura_Dataset'):
    try:
        from google.colab import drive
        if os.path.exists('/content/drive/MyDrive/Colab Notebooks/Basura_Dataset.zip'):
            shutil.copy('/content/drive/MyDrive/Colab Notebooks/Basura_Dataset.zip', '/content/Basura_Dataset.zip')
            os.system('unzip -q /content/Basura_Dataset.zip -d /content/')
        else:
            os.system('git clone https://github.com/abelcasv/clasificador_de_residuos.git /content/temp_repo')
            if os.path.exists('/content/temp_repo/Basura_Dataset'):
                shutil.move('/content/temp_repo/Basura_Dataset', '/content/Basura_Dataset')
                shutil.rmtree('/content/temp_repo')
    except ImportError:
        pass


## 1. Importación de Librerías y Fijación de Semillas
Configuración determinista para garantizar la reproducibilidad de todos los experimentos.


In [ ]:
import os
import copy
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from PIL import Image

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    os.system('pip install optuna -q')
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo de ejecucion: {device}")


## 2. Pipeline de Carga y Data Augmentation (SOLID - SRP)
Carga segura de imágenes descartando archivos corruptos o metadatos de sistema, y pipeline estocástico de aumento de datos.


In [ ]:
class SafeImageFolder(ImageFolder):
    def find_classes(self, directory):
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        classes = sorted(
            entry.name for entry in os.scandir(directory)
            if entry.is_dir() and not entry.name.startswith('.')
            and any(f.name.lower().endswith(valid_extensions) for f in os.scandir(entry.path) if f.is_file())
        )
        if not classes:
            raise FileNotFoundError(f"No se encontraron carpetas con imagenes validas en {directory}")
        class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
        return classes, class_to_idx

class DataPipeline:
    def __init__(self, data_dir='Basura_Dataset', resolucion=(224, 224), batch_size=32):
        self.data_dir = data_dir
        self.resolucion = resolucion
        self.batch_size = batch_size
        self._init_transforms()

    def _init_transforms(self):
        norm_mean = [0.485, 0.456, 0.406]
        norm_std = [0.229, 0.224, 0.225]

        self.train_transforms = transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm_mean, std=norm_std)
        ])

        self.eval_transforms = transforms.Compose([
            transforms.Resize(self.resolucion),
            transforms.ToTensor(),
            transforms.Normalize(mean=norm_mean, std=norm_std)
        ])

    def get_dataloaders(self):
        train_path = os.path.join(self.data_dir, 'train')
        val_path = os.path.join(self.data_dir, 'val')
        test_path = os.path.join(self.data_dir, 'test')

        ds_train = SafeImageFolder(train_path, transform=self.train_transforms)
        ds_val = SafeImageFolder(val_path, transform=self.eval_transforms)
        ds_test = SafeImageFolder(test_path, transform=self.eval_transforms)

        kwargs = {'num_workers': 2, 'pin_memory': True} if torch.cuda.is_available() else {}

        dl_train = DataLoader(ds_train, batch_size=self.batch_size, shuffle=True, **kwargs)
        dl_val = DataLoader(ds_val, batch_size=self.batch_size, shuffle=False, **kwargs)
        dl_test = DataLoader(ds_test, batch_size=self.batch_size, shuffle=False, **kwargs)

        return dl_train, dl_val, dl_test, ds_train.classes


## 3. Arquitectura Convolucional Modular Propia (WasteCNN)
Red convolucional de 3 etapas de extracción piramidal, agrupamiento MaxPool, Dropout y aplanamiento dinámico.


In [ ]:
class WasteCNN(nn.Module):
    def __init__(self, num_classes=7, input_size=96, fc_units=128, dropout_rate=0.25, use_leaky=False):
        super(WasteCNN, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, stride=1, padding=2)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.act = nn.LeakyReLU(negative_slope=0.1) if use_leaky else nn.ReLU()
        
        reduced_dim = input_size // (2 ** 3)
        flatten_dim = 64 * reduced_dim * reduced_dim
        
        self.fc1 = nn.Linear(flatten_dim, fc_units)
        self.fc2 = nn.Linear(fc_units, num_classes)

    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## 4. Módulo de Parada Anticipada y Orquestador de Entrenamiento
Gestión desacoplada del ciclo de vida del modelo y restauración automática del mejor estado en validación.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.best_epoch = 0
        self.early_stop = False
        self.best_weights = None

    def __call__(self, val_loss, model, epoch):
        if val_loss < (self.best_loss - self.min_delta):
            self.best_loss = val_loss
            self.best_epoch = epoch
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter = 0
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False

    def restore_best_weights(self, model):
        if self.best_weights is not None:
            model.load_state_dict(self.best_weights)

class ModelTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, device=device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device

    def train_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in self.train_loader:
            imgs, labels = imgs.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(imgs)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        return running_loss / total, correct / total

    def validate(self):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for imgs, labels in self.val_loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device)
                outputs = self.model(imgs)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item() * imgs.size(0)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        return running_loss / total, correct / total

    def fit(self, epochs=30, patience=5):
        early_stopping = EarlyStopping(patience=patience)
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

        start_time = time.time()
        for epoch in range(1, epochs + 1):
            train_loss, train_acc = self.train_epoch()
            val_loss, val_acc = self.validate()

            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            improved = early_stopping(val_loss, self.model, epoch)
            status = f"Epoca [{epoch:2d}/{epochs:2d}] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc*100:.2f}%"
            if improved:
                status += " (Mejora guardada)"
            print(status)

            if early_stopping.early_stop:
                print(f"Early Stopping activado en epoca {epoch}. Restaurando mejor epoca: {early_stopping.best_epoch}")
                break

        total_time = time.time() - start_time
        print(f"Entrenamiento completado en {total_time:.2f}s ({total_time/60:.2f} min).")
        early_stopping.restore_best_weights(self.model)
        return self.model, history, early_stopping.best_epoch


## 5. Módulo de Evaluación y Generación de Gráficas
Visualización dual de curvas (Loss y Accuracy) y matrices de confusión, con guardado automático en la carpeta results.


In [ ]:
class ModelEvaluator:
    @staticmethod
    def plot_dual_metrics(history, best_epoch, title='Curvas de Aprendizaje', save_path=None):
        epochs = range(1, len(history['train_loss']) + 1)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), dpi=150)

        ax1.plot(epochs, history['train_loss'], label='Train Loss', color='tab:blue', lw=2.2)
        ax1.plot(epochs, history['val_loss'], label='Val Loss', color='tab:orange', lw=2.2)
        if best_epoch:
            ax1.axvline(best_epoch, color='red', linestyle='--', lw=1.8, label=f'Mejor Epoca ({best_epoch})')
        ax1.set_title(f'{title} - Perdida', fontsize=12, fontweight='bold')
        ax1.set_xlabel('Epoca', fontsize=11)
        ax1.set_ylabel('Cross Entropy Loss', fontsize=11)
        ax1.legend(loc='upper right', fontsize=10)
        ax1.grid(True, linestyle=':', alpha=0.6)

        ax2.plot(epochs, [a * 100 for a in history['train_acc']], label='Train Acc', color='tab:green', lw=2.2)
        ax2.plot(epochs, [a * 100 for a in history['val_acc']], label='Val Acc', color='tab:purple', lw=2.2)
        if best_epoch:
            ax2.axvline(best_epoch, color='red', linestyle='--', lw=1.8, label=f'Mejor Epoca ({best_epoch})')
        ax2.set_title(f'{title} - Precision (Accuracy)', fontsize=12, fontweight='bold')
        ax2.set_xlabel('Epoca', fontsize=11)
        ax2.set_ylabel('Accuracy (%)', fontsize=11)
        ax2.legend(loc='lower right', fontsize=10)
        ax2.grid(True, linestyle=':', alpha=0.6)

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            print(f'Grafica guardada en: {save_path}')
        plt.show()

    @staticmethod
    def evaluate(model, test_loader, classes, device=device, save_matrix_path=None):
        model.eval()
        y_true, y_pred = [], []

        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                y_pred.extend(outputs.argmax(dim=1).cpu().numpy())
                y_true.extend(labels.cpu().numpy())

        acc = accuracy_score(y_true, y_pred)
        print(f'Accuracy Global en Test: {acc*100:.2f}%')
        print(classification_report(y_true, y_pred, target_names=classes, digits=4))

        cm = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(8, 6), dpi=150)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
        plt.title('Matriz de Confusion en Test', fontsize=12, fontweight='bold')
        plt.xlabel('Prediccion', fontsize=11)
        plt.ylabel('Etiqueta Real', fontsize=11)
        plt.tight_layout()
        if save_matrix_path:
            plt.savefig(save_matrix_path, dpi=200, bbox_inches='tight')
            print(f'Matriz guardada en: {save_matrix_path}')
        plt.show()
        return acc, cm


## 6. Fase 1: Búsqueda Automática de Hiperparámetros con Optuna (TPE)
Exploración bayesiana del espacio de hiperparámetros (LR, Dropout, neuronas FC, activación) sobre la red artesanal WasteCNN para verificar su techo empírico.


In [ ]:
class OptunaTuner:
    def __init__(self, train_loader, val_loader, input_size=96, num_classes=7):
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.input_size = input_size
        self.num_classes = num_classes

    def objective(self, trial):
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        dropout = trial.suggest_float('dropout', 0.15, 0.50, step=0.05)
        fc_units = trial.suggest_categorical('fc_units', [64, 128, 256])
        use_leaky = trial.suggest_categorical('use_leaky', [True, False])

        model = WasteCNN(
            num_classes=self.num_classes,
            input_size=self.input_size,
            fc_units=fc_units,
            dropout_rate=dropout,
            use_leaky=use_leaky
        ).to(device)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        best_val_acc = 0.0
        print(f"\n--- Ensayo Optuna [{trial.number + 1:2d}/10] | LR: {lr:.5f} | Drop: {dropout:.2f} | FC: {fc_units} | LeakyReLU: {use_leaky} ---")
        for epoch in range(1, 9):
            model.train()
            running_loss, total_train, correct_train = 0.0, 0, 0
            for imgs, labels in self.train_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * imgs.size(0)
                correct_train += (outputs.argmax(dim=1) == labels).sum().item()
                total_train += labels.size(0)

            train_loss = running_loss / total_train
            train_acc = correct_train / total_train

            model.eval()
            y_t, y_p = [], []
            with torch.no_grad():
                for imgs, labels in self.val_loader:
                    imgs, labels = imgs.to(device), labels.to(device)
                    outputs = model(imgs)
                    y_p.extend(outputs.argmax(dim=1).cpu().numpy())
                    y_t.extend(labels.cpu().numpy())

            val_acc = accuracy_score(y_t, y_p)
            improved_tag = ""
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                improved_tag = " (Mejor del ensayo)"
            
            print(f"  Ensayo [{trial.number + 1:2d}/10] | Epoca [{epoch}/8] | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}%{improved_tag}")
            
            trial.report(val_acc, epoch)
            if trial.should_prune():
                print(f"  Ensayo [{trial.number + 1:2d}/10] podado tempranamente por bajo rendimiento en epoca {epoch}.")
                raise optuna.exceptions.TrialPruned()

        print(f"--> Ensayo [{trial.number + 1:2d}/10] Completado. Mejor Val Acc: {best_val_acc*100:.2f}%")
        return best_val_acc

pipeline_optuna = DataPipeline(resolucion=(96, 96), batch_size=32)
dl_tr_opt, dl_va_opt, _, _ = pipeline_optuna.get_dataloaders()

tuner = OptunaTuner(dl_tr_opt, dl_va_opt, input_size=96)
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_warmup_steps=3))
study.optimize(tuner.objective, n_trials=10)

print("\n" + "=" * 60)
print(f"RESULTADO FINAL OPTUNA (10 Ensayos): Mejor Val Accuracy: {study.best_value*100:.2f}%")
print("Mejores Hiperparametros encontrados por Optuna:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print("=" * 60)


## 7. Fase 2: Transfer Learning con MobileNetV2 (Red Ligera)
Entrenamiento de la arquitectura convolucional eficiente MobileNetV2 basada en bloques residuales invertidos y convoluciones separables en profundidad.


In [ ]:
pipeline_mobile = DataPipeline(resolucion=(224, 224), batch_size=32)
dl_train, dl_val, dl_test, clases = pipeline_mobile.get_dataloaders()

modelo_mobile = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
modelo_mobile.classifier[1] = nn.Linear(modelo_mobile.last_channel, len(clases))
modelo_mobile = modelo_mobile.to(device)

criterio_mobile = nn.CrossEntropyLoss()
optimizador_mobile = optim.Adam(modelo_mobile.parameters(), lr=0.0005)

trainer_mobile = ModelTrainer(modelo_mobile, dl_train, dl_val, criterio_mobile, optimizador_mobile)
modelo_mobile, hist_mobile, mejor_ep_mobile = trainer_mobile.fit(epochs=30, patience=5)

ModelEvaluator.plot_dual_metrics(hist_mobile, mejor_ep_mobile, title='MobileNetV2', save_path='results/mobilenet_curvas.png')
acc_mobile, cm_mobile = ModelEvaluator.evaluate(modelo_mobile, dl_test, clases, save_matrix_path='results/matriz_mobilenet.png')


## 8. Fase 3: Transfer Learning con ResNet-18 (Modelo Ganador)
Entrenamiento de la arquitectura residual ResNet-18 para superar la cota del 85% y resolver definitivamente la ambigüedad espectral entre metal y vidrio.


In [ ]:
modelo_resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = modelo_resnet.fc.in_features
modelo_resnet.fc = nn.Linear(num_ftrs, len(clases))
modelo_resnet = modelo_resnet.to(device)

criterio_resnet = nn.CrossEntropyLoss()
optimizador_resnet = optim.Adam(modelo_resnet.parameters(), lr=0.0005)

trainer_resnet = ModelTrainer(modelo_resnet, dl_train, dl_val, criterio_resnet, optimizador_resnet)
modelo_resnet, hist_resnet, mejor_ep_resnet = trainer_resnet.fit(epochs=35, patience=5)

ModelEvaluator.plot_dual_metrics(hist_resnet, mejor_ep_resnet, title='ResNet-18', save_path='results/prueba_5_curvas.png')
acc_resnet, cm_resnet = ModelEvaluator.evaluate(modelo_resnet, dl_test, clases, save_matrix_path='results/matriz_prueba_5.png')


## 9. Fase 4: Comparativa Consolidada Final
Resumen comparativo entre las arquitecturas de transferencia evaluadas en el conjunto de test.


In [ ]:
print("=" * 60)
print(f"RESUMEN FINAL DE TRANSFER LEARNING EN TEST:")
print(f"  MobileNetV2 Test Accuracy: {acc_mobile*100:.2f}%")
print(f"  ResNet-18   Test Accuracy: {acc_resnet*100:.2f}%")
print("=" * 60)
dif = (acc_resnet - acc_mobile) * 100
if dif >= 0:
    print(f"ResNet-18 supera a MobileNetV2 por +{dif:.2f}% de precision.")
else:
    print(f"MobileNetV2 supera a ResNet-18 por +{-dif:.2f}% de precision.")
